# 03. Modelos Base y Evaluacion

Este notebook toma el dataset maestro generado en Notebook 02 y construye:
- baseline multiclase
- modelos base de clasificacion
- comparacion probabilistica por log loss
- modelo Poisson para distribucion de goles

In [40]:
import os
from pathlib import Path

os.environ["LOKY_MAX_CPU_COUNT"] = "1"

import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

modelingBasePath = Path("../data/intermediate/modeling_base_dataset.csv")
print("Modeling base path:", modelingBasePath)

Modeling base path: ..\data\intermediate\modeling_base_dataset.csv


## 1. Carga del dataset maestro

In [41]:
# =========================================================
# 1. CARGA DEL DATASET
# =========================================================
df = pd.read_csv(modelingBasePath)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date", kind="mergesort").reset_index(drop=True)

print("Shape df:", df.shape)
df.head()

Shape df: (25013, 78)


,date,homeTeam,awayTeam,tournament,neutral,matchMonth,matchYear,homeScore,awayScore,target,targetEncoded,isCloseMatch,isVeryCloseMatch,totalGoalsAvg,lowScoringSignal,lowTotalGoalsStrong,drawTendency,homeElo,awayElo,eloDiff,absEloDiff,eloExpectedHomeWin,homeMatchesPlayed,awayMatchesPlayed,matchesPlayedDiff,homeWinRate,awayWinRate,winRateDiff,homeDrawRate,awayDrawRate,drawRateDiff,homeLossRate,awayLossRate,lossRateDiff,homeGoalsForAvg,awayGoalsForAvg,goalsForAvgDiff,homeGoalsAgainstAvg,awayGoalsAgainstAvg,goalsAgainstAvgDiff,homeGoalDiffAvg,awayGoalDiffAvg,goalDiffAvgDiff,homeLast5WinRate,awayLast5WinRate,last5WinRateDiff,homeLast5PointsAvg,awayLast5PointsAvg,last5PointsDiff,homeLast5GoalsForAvg,awayLast5GoalsForAvg,last5GoalsForDiff,homeLast5GoalsAgainstAvg,awayLast5GoalsAgainstAvg,last5GoalsAgainstDiff,homeLast5GoalDiffAvg,awayLast5GoalDiffAvg,last5GoalDiffDiff,homeLast10PointsAvg,awayLast10PointsAvg,last10PointsDiff,homeLast10GoalsForAvg,awayLast10GoalsForAvg,last10GoalsForDiff,homeLast10GoalsAgainstAvg,awayLast10GoalsAgainstAvg,last10GoalsAgainstDiff,homeDaysSinceLastMatch,awayDaysSinceLastMatch,daysSinceLastMatchDiff,homeH2HWinRate,awayH2HWinRate,h2hDrawRate,tournamentCat_continental,tournamentCat_friendly,tournamentCat_other,tournamentCat_qualification,tournamentCat_worldCup
0,2000-01-04,Egypt,Togo,Friendly,0,1,2000,2,1,win,2,1,1,2.744185,0,0,215.335666,1500.0,1500.000000,0.000000,0.000000,0.592466,0,0,0,0.391509,0.378641,0.015171,0.23913,0.239437,0.0,0.352941,0.369492,-0.017047,1.392857,1.352113,0.043808,1.225806,1.258065,-0.047008,0.185185,0.121212,0.09632,0.4,0.4,0.0,1.4,1.4,0.0,1.2,1.2,0.0,1.2,1.2,0.0,0.0,0.0,0.0,1.4,1.4,0.0,1.3,1.3,0.0,1.2,1.2,0.0,12.0,12.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
1,2000-01-07,Tunisia,Togo,Friendly,0,1,2000,7,0,win,2,1,1,2.744185,0,0,215.335666,1500.0,1491.849325,8.150675,8.150675,0.603744,0,1,-1,0.391509,0.000000,0.015171,0.23913,0.000000,0.0,0.352941,1.000000,-0.017047,1.392857,1.000000,0.043808,1.225806,2.000000,-0.047008,0.185185,-1.000000,0.09632,0.4,0.0,0.0,1.4,0.0,0.0,1.2,1.0,0.0,1.2,2.0,0.0,0.0,-1.0,0.0,1.4,0.0,0.0,1.3,1.0,0.0,1.2,2.0,0.0,12.0,3.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
2,2000-01-08,Trinidad and Tobago,Canada,Friendly,0,1,2000,0,0,draw,1,1,1,2.744185,0,0,215.335666,1500.0,1500.000000,0.000000,0.000000,0.592466,0,0,0,0.391509,0.378641,0.015171,0.23913,0.239437,0.0,0.352941,0.369492,-0.017047,1.392857,1.352113,0.043808,1.225806,1.258065,-0.047008,0.185185,0.121212,0.09632,0.4,0.4,0.0,1.4,1.4,0.0,1.2,1.2,0.0,1.2,1.2,0.0,0.0,0.0,0.0,1.4,1.4,0.0,1.3,1.3,0.0,1.2,1.2,0.0,12.0,12.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
3,2000-01-09,Burkina Faso,Gabon,Friendly,0,1,2000,1,1,draw,1,1,1,2.744185,0,0,215.335666,1500.0,1500.000000,0.000000,0.000000,0.592466,0,0,0,0.391509,0.378641,0.015171,0.23913,0.239437,0.0,0.352941,0.369492,-0.017047,1.392857,1.352113,0.043808,1.225806,1.258065,-0.047008,0.185185,0.121212,0.09632,0.4,0.4,0.0,1.4,1.4,0.0,1.2,1.2,0.0,1.2,1.2,0.0,0.0,0.0,0.0,1.4,1.4,0.0,1.3,1.3,0.0,1.2,1.2,0.0,12.0,12.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0
4,2000-01-09,Guatemala,Armenia,Friendly,1,1,2000,1,1,draw,1,1,1,2.744185,0,0,215.335666,1500.0,1500.000000,0.000000,0.000000,0.500000,0,0,0,0.391509,0.378641,0.015171,0.23913,0.239437,0.0,0.352941,0.369492,-0.017047,1.392857,1.352113,0.043808,1.225806,1.258065,-0.047008,0.185185,0.121212,0.09632,0.4,0.4,0.0,1.4,1.4,0.0,1.2,1.2,0.0,1.2,1.2,0.0,0.0,0.0,0.0,1.4,1.4,0.0,1.3,1.3,0.0,1.2,1.2,0.0,12.0,12.0,0.0,0.333333,0.3,0.166667,0,1,0,0,0


## 2. Validacion y definicion de features

In [42]:
# =========================================================
# 2. VALIDACION Y DEFINICION DE FEATURES
# =========================================================
requiredColumns = [
    "date",
    "homeTeam",
    "awayTeam",
    "tournament",
    "homeScore",
    "awayScore",
    "target",
    "targetEncoded",
]

missingColumns = [col for col in requiredColumns if col not in df.columns]
assert not missingColumns, f"Faltan columnas requeridas: {missingColumns}"
assert df[requiredColumns].isnull().sum().sum() == 0, "Hay nulos en columnas criticas"

nonFeatureColumns = [
    "date",
    "homeTeam",
    "awayTeam",
    "tournament",
    "homeScore",
    "awayScore",
    "target",
    "targetEncoded",
]
featureColumns = [col for col in df.columns if col not in nonFeatureColumns]

X = df[featureColumns].copy()
y = df["targetEncoded"].copy()

nonNumericFeatureColumns = X.select_dtypes(exclude=[np.number]).columns.tolist()
assert not nonNumericFeatureColumns, f"Hay features no numericas: {nonNumericFeatureColumns}"

print("Numero de features:", len(featureColumns))
print("Distribucion target:")
print(df["target"].value_counts(normalize=True).round(4))

Numero de features: 70
Distribucion target:
target
win     0.4811
loss    0.2859
draw    0.2329
Name: proportion, dtype: float64


## 3. Split temporal

In [43]:
# =========================================================
# 3. SPLIT TEMPORAL
# =========================================================
trainEnd = int(len(df) * 0.70)
valEnd = trainEnd + int(len(df) * 0.15)

trainDf = df.iloc[:trainEnd].copy()
valDf = df.iloc[trainEnd:valEnd].copy()
testDf = df.iloc[valEnd:].copy()

X_train = trainDf[featureColumns].copy()
X_val = valDf[featureColumns].copy()
X_test = testDf[featureColumns].copy()

y_train = trainDf["targetEncoded"].copy()
y_val = valDf["targetEncoded"].copy()
y_test = testDf["targetEncoded"].copy()

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

Train: (17509, 70)
Val: (3751, 70)
Test: (3753, 70)


## 4. Baseline de referencia

In [44]:
# =========================================================
# 4. BASELINE
# =========================================================
majorityClass = y_train.value_counts().idxmax()
y_val_base = np.full_like(y_val, majorityClass)

print("BASELINE")
print("Accuracy:", accuracy_score(y_val, y_val_base))
print("Balanced Accuracy:", balanced_accuracy_score(y_val, y_val_base))
print("F1:", f1_score(y_val, y_val_base, average="macro"))

BASELINE
Accuracy: 0.48067182084777393
Balanced Accuracy: 0.3333333333333333
F1: 0.21642059776737488


## 5. Modelos base de clasificacion

In [45]:
# =========================================================
# 5. MODELOS DE CLASIFICACION
# =========================================================
# Raw LogisticRegression
rawLogModel = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, class_weight="balanced")
)
rawLogModel.fit(X_train, y_train)

# Calibrated LogisticRegression - Sigmoid
calLogModelSigmoid = CalibratedClassifierCV(
    estimator=make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced")
    ),
    method="sigmoid",
    cv=5
)
calLogModelSigmoid.fit(X_train, y_train)

# Calibrated LogisticRegression - Isotonic
calLogModelIsotonic = CalibratedClassifierCV(
    estimator=make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced")
    ),
    method="isotonic",
    cv=5
)
calLogModelIsotonic.fit(X_train, y_train)

# Random Forest
rfModel = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=1,
    class_weight="balanced_subsample",
)
rfModel.fit(X_train, y_train)

def metricRow(name, yTrue, yPred, yProba):
    return {
        "model": name,
        "accuracy": accuracy_score(yTrue, yPred),
        "balancedAccuracy": balanced_accuracy_score(yTrue, yPred),
        "f1": f1_score(yTrue, yPred, average="macro"),
        "logLoss": log_loss(yTrue, yProba, labels=[0, 1, 2]),
    }

# VALIDACION
y_val_pred_rawLog = rawLogModel.predict(X_val)
y_val_proba_rawLog = rawLogModel.predict_proba(X_val)

y_val_pred_calSigmoid = calLogModelSigmoid.predict(X_val)
y_val_proba_calSigmoid = calLogModelSigmoid.predict_proba(X_val)

y_val_pred_calIsotonic = calLogModelIsotonic.predict(X_val)
y_val_proba_calIsotonic = calLogModelIsotonic.predict_proba(X_val)

y_val_pred_rf = rfModel.predict(X_val)
y_val_proba_rf = rfModel.predict_proba(X_val)

validationResults = pd.DataFrame(
    [
        metricRow("RawLogisticRegression", y_val, y_val_pred_rawLog, y_val_proba_rawLog),
        metricRow("CalibratedLogitSigmoid", y_val, y_val_pred_calSigmoid, y_val_proba_calSigmoid),
        metricRow("CalibratedLogitIsotonic", y_val, y_val_pred_calIsotonic, y_val_proba_calIsotonic),
        metricRow("RandomForest", y_val, y_val_pred_rf, y_val_proba_rf),
    ]
).sort_values(["logLoss", "balancedAccuracy", "f1"], ascending=[True, False, False]).reset_index(drop=True)

validationResults

,model,accuracy,balancedAccuracy,f1,logLoss
0,CalibratedLogitSigmoid,0.607571,0.507550,0.454936,0.867388
1,CalibratedLogitIsotonic,0.607571,0.510991,0.461730,0.868488
2,RandomForest,0.581178,0.526236,0.518766,0.888864
3,RawLogisticRegression,0.567582,0.540347,0.538555,0.895580


## 6. Seleccion del mejor Modelo

In [46]:
# =========================================================
# 6. SELECCION DEL MEJOR CLASIFICADOR
# =========================================================
bestModelName = validationResults.iloc[0]["model"]

classifierMap = {
    "RawLogisticRegression": rawLogModel,
    "CalibratedLogitSigmoid": calLogModelSigmoid,
    "CalibratedLogitIsotonic": calLogModelIsotonic,
    "RandomForest": rfModel,
}

bestModel = classifierMap[bestModelName]

y_val_pred_best = bestModel.predict(X_val)
y_val_proba_best = bestModel.predict_proba(X_val)

y_test_pred_best = bestModel.predict(X_test)
y_test_proba_best = bestModel.predict_proba(X_test)

testBestResults = pd.DataFrame(
    [metricRow(bestModelName, y_test, y_test_pred_best, y_test_proba_best)]
)

print("BEST MODEL:", bestModelName)
print(classification_report(y_test, y_test_pred_best, zero_division=0))
testBestResults

BEST MODEL: CalibratedLogitSigmoid
              precision    recall  f1-score   support

           0       0.57      0.64      0.61      1115
           1       0.37      0.01      0.02       858
           2       0.63      0.87      0.73      1780

    accuracy                           0.61      3753
   macro avg       0.52      0.51      0.45      3753
weighted avg       0.55      0.61      0.53      3753



,model,accuracy,balancedAccuracy,f1,logLoss
0,CalibratedLogitSigmoid,0.605915,0.507935,0.451943,0.881642


# 7. Poisson goal model

In [47]:
# =========================================================
# 7. POISSON GOAL MODEL
# =========================================================
# Poisson usa las mismas features numéricas pero con un modelo distinto:
# predice goles esperados por equipo y luego deriva probs de win/draw/loss.

poissonScaler = StandardScaler()

X_train_p = poissonScaler.fit_transform(X_train)
X_val_p = poissonScaler.transform(X_val)
X_test_p = poissonScaler.transform(X_test)

poissonHome = PoissonRegressor(alpha=0.1, max_iter=1000)
poissonAway = PoissonRegressor(alpha=0.1, max_iter=1000)

poissonHome.fit(X_train_p, trainDf["homeScore"])
poissonAway.fit(X_train_p, trainDf["awayScore"])

lambdaHome_val = np.clip(poissonHome.predict(X_val_p), 0.01, None)
lambdaAway_val = np.clip(poissonAway.predict(X_val_p), 0.01, None)

lambdaHome_test = np.clip(poissonHome.predict(X_test_p), 0.01, None)
lambdaAway_test = np.clip(poissonAway.predict(X_test_p), 0.01, None)

In [48]:
def matchOutcomeProbabilities(lh, la, maxGoals=10):
    goals = np.arange(maxGoals + 1)
    pHome = poisson.pmf(goals, lh)
    pAway = poisson.pmf(goals, la)
    grid = np.outer(pHome, pAway)

    totalProbability = grid.sum()
    if totalProbability > 0:
        grid = grid / totalProbability

    return np.array([
        np.triu(grid, 1).sum(),  # loss
        np.trace(grid),          # draw
        np.tril(grid, -1).sum()   # win
    ])

def getPoissonProba(lhArr, laArr):
    return np.array([
        matchOutcomeProbabilities(lh, la)
        for lh, la in zip(lhArr, laArr)
    ])

y_val_proba_poisson = getPoissonProba(lambdaHome_val, lambdaAway_val)
y_test_proba_poisson = getPoissonProba(lambdaHome_test, lambdaAway_test)

y_val_pred_poisson = np.argmax(y_val_proba_poisson, axis=1)
y_test_pred_poisson = np.argmax(y_test_proba_poisson, axis=1)

poissonValidationResults = pd.DataFrame(
    [metricRow("PoissonGoalModel", y_val, y_val_pred_poisson, y_val_proba_poisson)]
)

poissonTestResults = pd.DataFrame(
    [metricRow("PoissonGoalModel", y_test, y_test_pred_poisson, y_test_proba_poisson)]
)

poissonValidationResults

,model,accuracy,balancedAccuracy,f1,logLoss
0,PoissonGoalModel,0.603306,0.506045,0.443199,0.860605


In [49]:
# =========================================================
# 7.1 CALIBRACION DEL UMBRAL DE EMPATE EN VALIDACION
# =========================================================

def probabilityMargin(probaRow):
    sortedProbs = np.sort(probaRow)
    return sortedProbs[-1] - sortedProbs[-2]

def gatedPredict(baseProba, poisProba, drawThreshold, marginThreshold):
    preds = []
    for classifierProba, poissonProba in zip(baseProba, poisProba):
        margin = probabilityMargin(classifierProba)
        if poissonProba[1] >= drawThreshold and margin <= marginThreshold:
            pred = 1
        else:
            pred = int(np.argmax(classifierProba))
        preds.append(pred)
    return np.array(preds)

drawThresholdGrid = np.linspace(0.15, 0.45, 13)
marginThresholdGrid = np.linspace(0.02, 0.20, 10)

bestDrawThreshold = 0.25
bestMarginThreshold = 0.08
bestScore = -np.inf

for dt in drawThresholdGrid:
    for mt in marginThresholdGrid:
        preds = gatedPredict(y_val_proba_best, y_val_proba_poisson, dt, mt)
        score = f1_score(y_val, preds, average="macro")
        if score > bestScore:
            bestScore = score
            bestDrawThreshold = dt
            bestMarginThreshold = mt

print("Best draw threshold:", bestDrawThreshold)
print("Best margin threshold:", bestMarginThreshold)
print("Base model used in gated ensemble:", bestModelName)

Best draw threshold: 0.25
Best margin threshold: 0.2
Base model used in gated ensemble: CalibratedLogitSigmoid


In [50]:
# =========================================================
# 7.2 REGLA GATEADA FINAL
# =========================================================
# Decision:
# - Si Poisson ve empate con suficiente probabilidad
#   y el clasificador base no está muy separado, forzar draw.
# - La probabilidad final para logloss se mantiene como la del mejor clasificador.
#   Eso preserva calibración global y usa Poisson como decisión auxiliar.

y_val_pred_gated = gatedPredict(
    y_val_proba_best,
    y_val_proba_poisson,
    bestDrawThreshold,
    bestMarginThreshold
)

y_test_pred_gated = gatedPredict(
    y_test_proba_best,
    y_test_proba_poisson,
    bestDrawThreshold,
    bestMarginThreshold
)

# Probabilidades del GatedEnsemble: mantenemos las del mejor clasificador.
# El gate modifica la clase, no la probabilidad.
y_val_proba_gated = y_val_proba_best.copy()
y_test_proba_gated = y_test_proba_best.copy()


In [51]:
# =========================================================
# 7.3 METRICAS DEL GATED ENSEMBLE
# =========================================================

gatedValidationResults = pd.DataFrame(
    [metricRow("GatedEnsemble", y_val, y_val_pred_gated, y_val_proba_gated)]
)

gatedTestResults = pd.DataFrame(
    [metricRow("GatedEnsemble", y_test, y_test_pred_gated, y_test_proba_gated)]
)

print("GATED ENSEMBLE (TEST)")
print("Accuracy:", accuracy_score(y_test, y_test_pred_gated))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_test_pred_gated))
print("F1:", f1_score(y_test, y_test_pred_gated, average="macro"))
print("LogLoss:", log_loss(y_test, y_test_proba_gated, labels=[0, 1, 2]))
print()
print(classification_report(y_test, y_test_pred_gated, zero_division=0))

GATED ENSEMBLE (TEST)
Accuracy: 0.5736743938182787
Balanced Accuracy: 0.5252507615660736
F1: 0.5301176434373097
LogLoss: 0.88164183800257

              precision    recall  f1-score   support

           0       0.64      0.49      0.56      1115
           1       0.31      0.35      0.33       858
           2       0.68      0.73      0.70      1780

    accuracy                           0.57      3753
   macro avg       0.54      0.53      0.53      3753
weighted avg       0.58      0.57      0.57      3753



# 9. COMPARACION FINAL

In [52]:
# =========================================================
# 8. COMPARACION FINAL
# =========================================================

comparisonDf = pd.concat(
    [
        validationResults.assign(split="validation"),
        poissonValidationResults.assign(split="validation"),
        gatedValidationResults.assign(split="validation"),
        testBestResults.assign(split="test"),
        poissonTestResults.assign(split="test"),
        gatedTestResults.assign(split="test"),
    ],
    ignore_index=True,
)

comparisonDf["split"] = pd.Categorical(
    comparisonDf["split"],
    categories=["validation", "test"],
    ordered=True,
)

comparisonDf = comparisonDf.sort_values(
    ["split", "logLoss", "balancedAccuracy", "f1"],
    ascending=[True, True, False, False]
).reset_index(drop=True)

comparisonDf

,model,accuracy,balancedAccuracy,f1,logLoss,split
0,PoissonGoalModel,0.603306,0.506045,0.443199,0.860605,validation
1,GatedEnsemble,0.577713,0.526184,0.533166,0.867388,validation
2,CalibratedLogitSigmoid,0.607571,0.507550,0.454936,0.867388,validation
3,CalibratedLogitIsotonic,0.607571,0.510991,0.461730,0.868488,validation
4,RandomForest,0.581178,0.526236,0.518766,0.888864,validation
5,RawLogisticRegression,0.567582,0.540347,0.538555,0.895580,validation
6,PoissonGoalModel,0.598987,0.503399,0.439976,0.877053,test
7,GatedEnsemble,0.573674,0.525251,0.530118,0.881642,test
8,CalibratedLogitSigmoid,0.605915,0.507935,0.451943,0.881642,test
